# Mini Project Klasifikasi: Analisis Sentimen Ulasan Produk Tokopedia

**Nama:** Asma Kurnia Rohmawati
**Divisi:** Artificial Intelligence | **Program:** VINIX7 Magang Fast Track
**Keterangan:** Assignment Pengganti, Tugas 2 (Mini Project Klasifikasi)

**Cara menjalankan:** buka di Google Colab, lalu pilih *Runtime > Run all*. Notebook ini membutuhkan file dataset `PRDECT-ID_Dataset` (format .md atau .csv). Kalau file belum ada di folder kerja, sel pemuatan data akan membuka jendela unggah file.

## 1. Deskripsi Masalah

**Latar belakang.** Penjual di marketplace menerima ulasan dalam jumlah besar dan tidak mungkin membacanya satu per satu. Padahal ulasan negatif (barang rusak, tidak sampai, tidak sesuai deskripsi) perlu ditindaklanjuti dengan cepat. Klasifikasi sentimen otomatis bisa membantu memilah ulasan mana yang harus diprioritaskan. Kasus ini juga dekat dengan pekerjaan analitik e-commerce di bisnis digital.

**Tujuan.** Membangun model yang membaca teks ulasan berbahasa Indonesia lalu menebak sentimennya (positif atau negatif), kemudian membandingkan tiga algoritma dan memilih yang terbaik.

| Aspek | Keterangan |
|---|---|
| Sumber dataset | PRDECT-ID (*Product Reviews Dataset for Emotions Classification Tasks, Indonesian*), ulasan produk di Tokopedia, oleh Sutoyo dkk. (2022). Tersedia di Mendeley Data, DOI 10.17632/574v66hf2v.1 |
| Isi | Ulasan dari 29 kategori produk. Jumlah barisnya saya hitung langsung di sel pemuatan data |
| Target yang diprediksi | `Sentiment`: **Negative = 1** (kelas positif untuk metrik), Positive = 0 |
| Fitur | Teks `Customer Review` saja. Alasan kolom lain tidak dipakai dijelaskan di bagian 2 |
| Jenis masalah | Klasifikasi biner pada teks (supervised learning) |
| Algoritma | Logistic Regression, Random Forest, Multinomial Naive Bayes |
| Metrik | F1-score kelas negatif sebagai patokan utama, dilengkapi accuracy, precision, dan recall |

**Kenapa ulasan negatif dijadikan kelas positif.** Pada metrik, kelas yang ingin ditemukan disebut kelas positif. Yang ingin ditemukan penjual adalah ulasan negatif, sama seperti contoh deployment di Tugas 1. Jadi di notebook ini: TP = ulasan negatif yang berhasil tertangkap, FN = ulasan negatif yang terlewat, FP = ulasan positif yang salah ditandai negatif.

In [ ]:
# Import library yang dipakai di seluruh notebook
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE = 42          # supaya hasil bisa diulang dengan angka yang sama
pd.set_option("display.max_colwidth", 120)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

In [ ]:
# Memuat dataset PRDECT-ID. File .md berisi tabel dengan pemisah "|", file .csv dibaca biasa.
def baca_tabel_markdown(path):
    with open(path, encoding="utf-8") as f:
        baris = f.read().splitlines()
    header = [k.strip() for k in baris[0].strip().strip("|").split("|")]
    isi, dilewati = [], 0
    for b in baris[2:]:                       # baris ke-2 hanya garis pemisah header
        sel = [s.strip() for s in b.strip().strip("|").split("|")]
        if len(sel) == len(header):
            isi.append(sel)
        else:
            dilewati += 1                     # baris dengan jumlah kolom tidak cocok dilewati
    print("Baris yang dilewati karena format rusak:", dilewati)
    return pd.DataFrame(isi, columns=header)

def cari_file():
    # Cari file apa pun yang namanya memuat "PRDECT" (awalan/akhiran dari unduhan tidak masalah)
    kandidat = glob.glob("*PRDECT*") + glob.glob("/content/*PRDECT*")
    return sorted({f for f in kandidat if os.path.isfile(f) and f.lower().endswith((".md", ".csv"))})

def muat_data():
    file_data = cari_file()
    if not file_data:
        # Di Colab: minta pengguna mengunggah file dataset
        try:
            from google.colab import files
            files.upload()
            file_data = cari_file()
        except ImportError:
            pass
    if not file_data:
        raise FileNotFoundError("File PRDECT-ID_Dataset belum ada di folder kerja. "
                                "Letakkan file di folder yang sama dengan notebook, lalu jalankan ulang.")
    path = file_data[0]
    print("Membaca file:", path)
    return pd.read_csv(path) if path.lower().endswith(".csv") else baca_tabel_markdown(path)

mentah = muat_data()

# Ganti nama kolom ke bahasa Indonesia yang singkat, lalu ubah rating jadi angka
df = mentah.rename(columns={"Category": "kategori", "Customer Review": "ulasan",
                            "Customer Rating": "rating", "Sentiment": "sentimen",
                            "Emotion": "emosi"})[["kategori", "ulasan", "rating", "sentimen", "emosi"]].copy()
df["rating"] = df["rating"].astype(int)

print("Ukuran data (baris, kolom):", df.shape)
df.head()

## 2. Eksplorasi Data (EDA)

Bagian ini untuk mengenali data sebelum dimodelkan: jumlah data, kualitas data (nilai kosong dan duplikat), sebaran kelas, ciri teks, dan **kolom mana yang tidak boleh dipakai sebagai fitur**.

In [ ]:
# Gambaran umum: jumlah data, nilai kosong, dan duplikat
print("Jumlah ulasan:", len(df))
print("Jumlah kategori produk:", df["kategori"].nunique())
print("\nNilai kosong per kolom:")
print(df.isna().sum().to_string())
print("Teks ulasan kosong (string kosong):", (df["ulasan"].str.strip() == "").sum())
print("\nUlasan duplikat (isi teks sama):", df.duplicated(subset="ulasan").sum())

# Distribusi kelas target (jumlah dan persentase)
distribusi = df["sentimen"].value_counts()
persen = (df["sentimen"].value_counts(normalize=True) * 100).round(2)
print("\nDistribusi sentimen:")
print(pd.DataFrame({"jumlah": distribusi, "persen (%)": persen}).to_string())

In [ ]:
# Visualisasi 1: distribusi kelas sentimen
fig, ax = plt.subplots(figsize=(5, 4))
warna = {"Negative": "#E45756", "Positive": "#4C78A8"}
batang = ax.bar(distribusi.index, distribusi.values, color=[warna[k] for k in distribusi.index])
for b in batang:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(), str(int(b.get_height())),
            ha="center", va="bottom")
ax.set_title("Distribusi sentimen ulasan")
ax.set_ylabel("Jumlah ulasan")
plt.tight_layout()
plt.show()

Kedua kelas jumlahnya hampir sama, jadi data ini **relatif seimbang**. Artinya accuracy di sini cukup bisa dipercaya, berbeda dengan kasus data yang sangat timpang seperti fraud di Tugas 1. Meski begitu precision, recall, dan F1 tetap dilaporkan karena jenis kesalahannya punya dampak bisnis yang berbeda.

In [ ]:
# Visualisasi 2: hubungan rating bintang dengan sentimen
tab_rating = pd.crosstab(df["rating"], df["sentimen"])
tab_rating.plot(kind="bar", stacked=True, figsize=(7, 4), rot=0,
                color=[warna[k] for k in tab_rating.columns])
plt.title("Rating bintang vs sentimen")
plt.xlabel("Rating pelanggan")
plt.ylabel("Jumlah ulasan")
plt.tight_layout()
plt.show()
tab_rating

In [ ]:
# Cek kebocoran data: hubungan kolom emosi dan rating dengan target
print("Emosi vs sentimen:")
print(pd.crosstab(df["emosi"], df["sentimen"]).to_string())

# Uji sederhana: tebak "Negative" bila rating <= 3, tanpa membaca teks sama sekali
tebak_rating = np.where(df["rating"] <= 3, "Negative", "Positive")
print("\nAkurasi aturan 'rating <= 3 berarti negatif' (tanpa teks):",
      round((tebak_rating == df["sentimen"]).mean(), 4))

**Temuan penting: ada dua kolom yang membocorkan jawaban.**
- `emosi`: Anger, Fear, dan Sadness hanya muncul pada ulasan negatif, sedangkan Happy dan Love hanya pada ulasan positif. Kolom ini tidak mungkin diketahui sebelum sentimennya diketahui, jadi memakainya berarti model sudah "diberi tahu" jawabannya.
- `rating`: rating 1 dan 2 seluruhnya negatif, rating 4 dan 5 hampir seluruhnya positif. Aturan sederhana berdasarkan rating saja sudah menghasilkan akurasi sangat tinggi (lihat keluaran di atas), padahal tidak ada teks yang dibaca.

Tujuan project ini adalah memahami sentimen **dari teks ulasan**, jadi kedua kolom itu **tidak dipakai sebagai fitur**. Hanya kolom `ulasan` yang masuk ke model. Kolom lain (harga, lokasi, jumlah terjual, dan sebagainya) juga tidak dipakai agar fokusnya tetap pada teks.

In [ ]:
# Visualisasi 3: panjang ulasan (jumlah kata) per kelas
df["jumlah_kata"] = df["ulasan"].str.split().str.len()
batas = df["jumlah_kata"].quantile(0.99)          # potong 1% ulasan terpanjang agar grafik terbaca

fig, ax = plt.subplots(figsize=(7, 4))
for nama in ["Negative", "Positive"]:
    ax.hist(df.loc[(df["sentimen"] == nama) & (df["jumlah_kata"] <= batas), "jumlah_kata"],
            bins=30, alpha=0.6, density=True, label=nama, color=warna[nama])
ax.set_title("Sebaran panjang ulasan per kelas")
ax.set_xlabel("Jumlah kata")
ax.set_ylabel("Kepadatan")
ax.legend()
plt.tight_layout()
plt.show()

print("Jumlah kata per ulasan:")
print(df.groupby("sentimen")["jumlah_kata"].agg(["mean", "median"]).round(1).to_string())

## 3. Preprocessing

| Langkah | Alasan |
|---|---|
| Cek dan buang nilai kosong | Baris tanpa teks atau label tidak bisa dipakai belajar |
| Buang ulasan duplikat | Ulasan yang sama bisa jatuh ke data latih dan data uji sekaligus, sehingga skor tampak lebih bagus dari kenyataan |
| Encoding target (Negative = 1, Positive = 0) | Model hanya menerima angka, dan ulasan negatif dijadikan kelas positif sesuai tujuan |
| Text cleaning | Huruf kecil, buang tanda baca dan karakter rusak, rapikan huruf berulang ("mantappp"), dan ubah kata singkatan umum ("gk", "yg") ke bentuk bakunya |
| **Stopword tidak dibuang** | Kata seperti "tidak", "belum", dan "kurang" justru membawa sentimen. Membuangnya bisa mengubah arti ulasan |
| TF-IDF (unigram dan bigram) | Mengubah teks menjadi angka. Bigram membantu menangkap pasangan kata seperti "tidak sesuai". Dipasang di dalam pipeline supaya hanya belajar dari data latih |
| Scaling | Tidak perlu `StandardScaler`: nilai TF-IDF sudah dinormalisasi per dokumen, dan Random Forest tidak sensitif terhadap skala |

In [ ]:
# Missing value: buang baris tanpa ulasan atau tanpa label (jika ada)
sebelum = len(df)
df = df.dropna(subset=["ulasan", "sentimen"])
df = df[df["ulasan"].str.strip() != ""]
print("Baris dibuang karena kosong:", sebelum - len(df))

# Cek apakah ulasan yang sama punya label berbeda (konflik label)
konflik = (df.groupby("ulasan")["sentimen"].nunique() > 1).sum()
print("Teks ulasan dengan label bertentangan:", konflik)

# Duplikat: simpan kemunculan pertama saja
sebelum = len(df)
df = df.drop_duplicates(subset="ulasan", keep="first")
print("Baris dibuang karena duplikat:", sebelum - len(df))

# Encoding target: Negative = 1 (kelas yang ingin ditemukan), Positive = 0
df["target"] = df["sentimen"].map({"Negative": 1, "Positive": 0})
print("Label yang tidak terpetakan:", df["target"].isna().sum())

In [ ]:
# Text cleaning
# Kamus singkatan sederhana. Sengaja kecil dan hanya berisi singkatan yang artinya jelas.
KAMUS_SINGKATAN = {
    "gk": "tidak", "ga": "tidak", "gak": "tidak", "ngga": "tidak", "nggak": "tidak", "tdk": "tidak",
    "yg": "yang", "bgt": "banget", "udh": "sudah", "udah": "sudah", "sdh": "sudah",
    "dgn": "dengan", "krn": "karena", "utk": "untuk", "jgn": "jangan", "tp": "tapi",
    "blm": "belum", "lbh": "lebih", "jg": "juga", "klo": "kalau", "kalo": "kalau", "sy": "saya",
}

def bersihkan(teks):
    teks = teks.lower()                                   # huruf kecil semua
    teks = re.sub(r"http\S+|www\.\S+", " ", teks)         # buang tautan
    teks = re.sub(r"[^a-z\s]", " ", teks)                 # buang angka, tanda baca, emoji (muncul sebagai "?")
    teks = re.sub(r"(.)\1{2,}", r"\1", teks)              # huruf berulang 3x atau lebih jadi 1 ("mantappp" -> "mantap")
    kata = [KAMUS_SINGKATAN.get(k, k) for k in teks.split()]   # ubah singkatan ke bentuk baku
    return " ".join(kata)

df["ulasan_bersih"] = df["ulasan"].apply(bersihkan)

# Ulasan yang jadi kosong setelah dibersihkan (misalnya hanya berisi angka atau emoji) dibuang
sebelum = len(df)
df = df[df["ulasan_bersih"].str.len() > 0]
print("Baris dibuang karena kosong setelah cleaning:", sebelum - len(df))
print("Data siap dipakai:", len(df), "ulasan")
print(df["sentimen"].value_counts().to_string())

# Bandingkan teks asli dan teks bersih pada 5 contoh acak
contoh = df.sample(5, random_state=RANDOM_STATE)
for _, baris in contoh.iterrows():
    print(f"\n[{baris['sentimen']}] ASLI  : {baris['ulasan']}")
    print(f"           BERSIH: {baris['ulasan_bersih']}")

In [ ]:
# Visualisasi 4: 12 kata paling sering di tiap kelas setelah cleaning.
# Daftar kata umum di bawah hanya untuk grafik ini. Saat modeling, tidak ada stopword yang dibuang.
KATA_UMUM = ["yang", "dan", "di", "ke", "dari", "ini", "itu", "untuk", "dengan", "saya", "juga",
             "sudah", "ada", "pada", "karena", "jadi", "atau", "aja", "sih", "ya", "nya", "lagi",
             "sama", "kalau", "saja", "sampai", "bisa", "mau", "buat"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, nama in zip(axes, ["Negative", "Positive"]):
    cv = CountVectorizer(stop_words=KATA_UMUM)
    mat = cv.fit_transform(df.loc[df["sentimen"] == nama, "ulasan_bersih"])
    jumlah = np.asarray(mat.sum(axis=0)).ravel()
    urut = jumlah.argsort()[::-1][:12]
    kata = np.array(cv.get_feature_names_out())[urut]
    ax.barh(kata[::-1], jumlah[urut][::-1], color=warna[nama])
    ax.set_title(f"Kata teratas: {nama}")
    ax.set_xlabel("Frekuensi")
plt.tight_layout()
plt.show()

## 4. Modeling

Data dibagi **80% latih dan 20% uji** dengan `stratify=y` supaya proporsi kelas di kedua bagian sama. Tiga algoritma dibandingkan pada data yang sama:

- **Logistic Regression**: model linear yang biasanya menjadi baseline kuat untuk teks, dan koefisiennya bisa dibaca untuk melihat kata mana yang berpengaruh.
- **Random Forest**: kumpulan pohon keputusan. Saya masukkan sebagai pembanding dari keluarga model yang berbeda.
- **Multinomial Naive Bayes**: model probabilistik klasik untuk teks, sangat cepat dan sederhana.

Tiap model dibungkus `Pipeline` bersama TF-IDF, sehingga TF-IDF hanya dilatih dari data latih dan tidak melihat data uji.

In [ ]:
# Pisahkan fitur (teks bersih) dan target, lalu bagi 80:20
X = df["ulasan_bersih"]
y = df["target"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print("Data latih:", len(X_train), "| Data uji:", len(X_test))
print("Proporsi ulasan negatif di data latih: {:.2%}".format(y_train.mean()))
print("Proporsi ulasan negatif di data uji  : {:.2%}".format(y_test.mean()))

In [ ]:
# Definisi tiga pipeline: TF-IDF (unigram + bigram) lalu model
def buat_tfidf():
    # sublinear_tf meredam pengaruh kata yang diulang banyak kali dalam satu ulasan
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)

pipelines = {
    "Logistic Regression": Pipeline([
        ("tfidf", buat_tfidf()),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        ("tfidf", buat_tfidf()),
        ("clf", RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)),
    ]),
    "Naive Bayes": Pipeline([
        ("tfidf", buat_tfidf()),
        ("clf", MultinomialNB()),
    ]),
}

# Latih tiap model pada data latih, lalu prediksi data uji
prediksi = {}
for nama, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    prediksi[nama] = pipe.predict(X_test)
    print("Selesai dilatih:", nama)

## 5. Evaluasi

Semua metrik dihitung pada **data uji** dengan ulasan negatif sebagai kelas positif. Rumusnya sama seperti di Tugas 1: precision = TP / (TP + FP), recall = TP / (TP + FN), F1 = rata-rata harmonik keduanya.

In [ ]:
# Tabel metrik untuk tiap model
def hitung_metrik(y_asli, y_pred):
    return {
        "Accuracy": accuracy_score(y_asli, y_pred),
        "Precision": precision_score(y_asli, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_asli, y_pred, pos_label=1, zero_division=0),
        "F1-score": f1_score(y_asli, y_pred, pos_label=1, zero_division=0),
    }

hasil = pd.DataFrame({nama: hitung_metrik(y_test, p) for nama, p in prediksi.items()}).T.round(4)
hasil = hasil.sort_values("F1-score", ascending=False)
hasil

In [ ]:
# Perbandingan metrik dalam grafik batang
ax = hasil.plot(kind="bar", figsize=(9, 4.5), ylim=(0, 1.05), rot=0, width=0.8)
ax.set_title("Perbandingan metrik antar model (kelas negatif)")
ax.set_ylabel("Skor")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix tiap model (baris = label asli, kolom = tebakan model)
fig, axes = plt.subplots(1, len(prediksi), figsize=(5 * len(prediksi), 4))
for ax, (nama, pred) in zip(axes, prediksi.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=["Positive", "Negative"], cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(nama)
    ax.grid(False)
plt.tight_layout()
plt.show()

# Rincian empat angka confusion matrix dalam bentuk tabel
rincian = []
for nama, pred in prediksi.items():
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rincian.append({"Model": nama,
                    "TP (negatif tertangkap)": tp,
                    "TN (positif benar)": tn,
                    "FP (positif dikira negatif)": fp,
                    "FN (negatif terlewat)": fn})
pd.DataFrame(rincian).set_index("Model")

Cara membaca: **FN** berarti ulasan negatif yang lolos tanpa diprioritaskan, sehingga keluhan pelanggan bisa telat ditangani. **FP** berarti ulasan positif yang salah ditandai negatif, sehingga penjual membuang waktu mengecek ulasan yang sebenarnya aman. Untuk kasus ini FN umumnya lebih merugikan, jadi recall kelas negatif perlu diperhatikan, tetapi precision tetap dijaga supaya daftar ulasan yang ditandai tidak penuh salah alarm.

In [ ]:
# Analisis kesalahan pada model terbaik (urut F1, lalu recall bila seri)
terbaik = hasil.sort_values(["F1-score", "Recall"], ascending=False).index[0]
pred_terbaik = prediksi[terbaik]
print("Model terbaik:", terbaik)

salah = df.loc[X_test.index[y_test.values != pred_terbaik], ["sentimen", "rating", "ulasan"]]
fn_df = salah[salah["sentimen"] == "Negative"]
fp_df = salah[salah["sentimen"] == "Positive"]

print("\nUlasan negatif yang terlewat (FN):", len(fn_df))
print(fn_df.head(5)[["rating", "ulasan"]].to_string(index=False))
print("\nUlasan positif yang salah ditandai negatif (FP):", len(fp_df))
print(fp_df.head(5)[["rating", "ulasan"]].to_string(index=False))

In [ ]:
# Interpretasi Logistic Regression: koefisien positif mendorong ke kelas negatif, koefisien negatif ke positif
tfidf_lr = pipelines["Logistic Regression"].named_steps["tfidf"]
koef = pipelines["Logistic Regression"].named_steps["clf"].coef_[0]
kata = np.array(tfidf_lr.get_feature_names_out())
urut = np.argsort(koef)
print("12 kata/frasa pendorong NEGATIF:", list(kata[urut[-12:][::-1]]))
print("12 kata/frasa pendorong POSITIF:", list(kata[urut[:12]]))

## 6. Kesimpulan

In [ ]:
# Ringkasan hasil dibuat otomatis dari tabel evaluasi supaya selalu cocok dengan angka terbaru
b = hasil.loc[terbaik]
print(f"Model terbaik berdasarkan F1-score kelas negatif: {terbaik}")
print(f"  Accuracy {b['Accuracy']:.4f} | Precision {b['Precision']:.4f} | "
      f"Recall {b['Recall']:.4f} | F1 {b['F1-score']:.4f}")
for nama in hasil.index:
    if nama != terbaik:
        r = hasil.loc[nama]
        print(f"  Dibanding {nama}: selisih F1 {b['F1-score'] - r['F1-score']:+.4f}, "
              f"precision {b['Precision'] - r['Precision']:+.4f}, recall {b['Recall'] - r['Recall']:+.4f}")

**Model terbaik: Logistic Regression.** Pada data uji, model ini mencatat F1-score kelas negatif sekitar 0,96, tertinggi dari ketiga model, dengan recall sekitar 0,99. Artinya dari sekitar 550 ulasan negatif di data uji, hanya 8 yang terlewat. Naive Bayes menyusul dengan selisih tipis (F1 sekitar 0,95), sedangkan Random Forest paling rendah (F1 sekitar 0,94) dan paling banyak melewatkan ulasan negatif. Angka persisnya mengikuti keluaran sel di atas.

**Alasan yang masuk akal.**
- Setelah TF-IDF dengan bigram, teks menjadi matriks yang sangat lebar dan sebagian besar nilainya nol. Model linear cocok untuk kondisi ini karena memberi bobot pada tiap kata atau frasa, dan sinyal sentimen di data ini memang kuat pada kata tertentu. Hal itu terlihat dari koefisien Logistic Regression: "tidak", "kurang", "kecewa", dan "rusak" mendorong ke negatif, sedangkan "bagus", "mantap", dan "cepat" mendorong ke positif.
- Random Forest kemungkinan kurang diuntungkan karena tiap pohonnya hanya menguji sebagian kecil dari ribuan fitur yang jarang muncul. Ini dugaan saya dari pola hasilnya, belum saya uji terpisah.
- Selisih Logistic Regression dan Naive Bayes kurang dari satu poin F1. Dengan satu kali pembagian data, saya tidak bisa memastikan keduanya benar-benar berbeda.

**Hal yang perlu diingat saat membaca hasil.**
- Kolom `emosi` dan `rating` sengaja tidak dipakai karena membocorkan jawaban. Kalau dipakai, skor akan terlihat sangat tinggi tetapi tidak menunjukkan kemampuan membaca teks.
- Kesalahan model umumnya terjadi pada ulasan campuran (memuji sekaligus mengeluh), ejaan atau singkatan yang belum ada di kamus (misalnya "g" untuk "tidak"), dan emoji yang hilang saat cleaning.
- Evaluasi memakai satu kali pembagian 80:20, jadi skor bisa bergeser sedikit kalau `random_state` diganti.
- Data hanya berisi ulasan Tokopedia dengan dua kelas (tanpa netral), sehingga hasilnya belum tentu berlaku di platform lain atau untuk ulasan yang netral.

**Dua ide perbaikan.**
1. **Evaluasi dan tuning yang lebih kuat.** Pakai stratified k-fold cross-validation dan `GridSearchCV` untuk mencari pengaturan terbaik (misalnya nilai `C` pada Logistic Regression, `ngram_range`, dan `min_df`), serta perluas kamus singkatan atau tambahkan stemming bahasa Indonesia agar variasi ejaan tidak dihitung sebagai kata berbeda.
2. **Model bahasa berbahasa Indonesia.** Fine-tuning model seperti IndoBERT, yang membaca konteks seluruh kalimat, berpotensi lebih baik menangani ulasan campuran yang sulit bagi model berbasis TF-IDF. Konsekuensinya butuh komputasi lebih besar (GPU di Colab).

## Catatan Penggunaan AI dan Referensi

**Catatan:** Notebook ini disusun dengan bantuan Claude (Anthropic) untuk kerangka kode dan merapikan struktur. Setiap sel sudah saya jalankan, baca, dan pahami, dan saya siap menjelaskannya.

**Referensi**
1. Sutoyo, R., Chowanda, A., Achmad, S., Andangsari, E. W., Isa, S. M., dkk. (2022). *Product Reviews Dataset for Emotions Classification Tasks - Indonesian (PRDECT-ID) Dataset.* Mendeley Data, V1. https://doi.org/10.17632/574v66hf2v.1
2. Sutoyo, R., dkk. (2022). *PRDECT-ID: Indonesian product reviews dataset for emotions classification tasks.* Data in Brief.
3. scikit-learn developers. *Working With Text Data.* https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html
4. scikit-learn developers. *Metrics and scoring: quantifying the quality of predictions.* https://scikit-learn.org/stable/modules/model_evaluation.html
5. Google for Developers. *Classification: Accuracy, recall, precision, and related metrics* (Machine Learning Crash Course). https://developers.google.com/machine-learning/crash-course/classification/accuracy-precision-recall